# Powerlifting Strength Analysis

An exploratory data analysis and interactive visualization project examining how bodyweight relates to absolute strength, where total strength shows diminishing returns, and how the three competition lifts contribute to total performance among Raw USAPL lifters.


In [21]:
# Import required packages
import pandas as pd
import numpy as np
import altair as alt


In [13]:
# Load the cleaned USAPL dataset
df = pd.read_csv("openpowerlifting_usapl_clean.csv")

# Restrict the final analysis to male and female lifters.
# The Mx category was excluded because its sample size was too small
# to support stable subgroup comparisons in this project.
df = df[df["Sex"].isin(["M", "F"])].copy()

# Reset the index after filtering and verify the final sample size.
df.reset_index(drop=True, inplace=True)

print(df["Sex"].value_counts())
print("\nDataset shape:", df.shape)


Sex
M    62190
F    31300
Name: count, dtype: int64

Dataset shape: (93490, 16)


In [14]:
df.head()

,Name,Sex,Event,Equipment,Age,BodyweightKg,WeightClassKg,Squat,Bench,Deadlift,Total,Place,Dots,Tested,Federation,Date
0,Devonte Lewis #1,M,SBD,Raw,27.5,213.55,140+,490.5,288.5,357.5,1136.5,1,563.27,Yes,USAPL,2026-06-12
1,Ray Williams #1,M,SBD,Raw,32.0,190.40,120+,490.0,247.5,375.0,1112.5,1,562.77,Yes,USAPL,2019-03-02
2,Frank Allen #1,M,SBD,Raw,33.0,175.30,140+,422.5,272.5,375.0,1070.0,1,552.46,Yes,USAPL,2023-05-20
3,Jesus Olivares,M,SBD,Raw,22.0,157.80,120+,410.0,245.5,400.0,1055.5,1,560.02,Yes,USAPL,2020-12-12
4,Rondel Hunte,M,SBD,Raw,27.0,119.54,125,385.0,262.5,397.5,1045.0,G,600.92,Yes,USAPL,2023-09-14


## Project Overview

### Objective

This project uses OpenPowerlifting meet-result data to investigate three questions:

1. **Which competition lift scales most strongly with bodyweight?**
2. **At what bodyweight does total strength begin to show diminishing returns?**
3. **How do squat, bench press, and deadlift proportions differ between male and female lifters?**

### Analytical population

The source data contained approximately **4.0 million meet-result records**. The analysis was restricted to **Raw, full-power USAPL performances** with valid squat, bench press, deadlift, total, bodyweight, age, weight class, and sex information. Disqualified performances were removed.

To reduce repeated observations, the analysis retains one best performance for each matching **Name–Sex** combination after the earlier cleaning process. The final analytical dataset contains approximately **93,000 male and female lifters**.

### Tools

- **Python**
- **pandas** for data manipulation
- **NumPy** for numerical analysis and regression calculations
- **Altair** for interactive visualization

### Design approach

The project progresses from individual lifts, to overall total strength, to the composition of that total. Interactive filters are used where subgroup comparisons materially change interpretation.


## Task 1: Which lift scales best with bodyweight?

We first examine the relationship between bodyweight and the squat, bench press, and deadlift. Correlation measures the strength of the linear relationship between bodyweight and each lift.


In [15]:
# Calculate Pearson correlations between bodyweight and each competition lift.
# The correlations are sorted from strongest to weakest.
bodyweight_correlations = (
    df[["BodyweightKg", "Squat", "Bench", "Deadlift"]]
    .corr()["BodyweightKg"]
    .drop("BodyweightKg")
    .sort_values(ascending=False)
)

bodyweight_correlations


Bench       0.596694
Squat       0.593944
Deadlift    0.551530
Name: BodyweightKg, dtype: float64

### Overall Correlation with Bodyweight

The results show a moderate positive correlation between bodyweight and all three competition lifts. Bench press has the strongest correlation with bodyweight ($r = 0.597$), followed very closely by squat ($r = 0.594$). Deadlift has the weakest relationship of the three ($r = 0.552$).

These results suggest that heavier lifters generally lift more weight across all three movements. However, the difference between bench press and squat is very small, indicating that their relationships with bodyweight are similar. Deadlift appears to be somewhat less strongly associated with increases in bodyweight.

It is important to note that correlation measures the strength of the linear relationship, but does not tell us how much each lift increases for each additional kilogram of bodyweight. We will examine this relationship further using regression slopes and visualization.

In [17]:
# Recalculate the bodyweight correlations within each sex.
# This checks whether the pooled relationship changes after stratification.
sex_correlations = (
    df.groupby("Sex")
      .apply(
          lambda group: group[["BodyweightKg", "Squat", "Bench", "Deadlift"]]
          .corr()["BodyweightKg"]
          .drop("BodyweightKg"),
          include_groups=False
      )
)

sex_correlations


BodyweightKg,Squat,Bench,Deadlift
Sex,,,
F,0.398922,0.374565,0.361772
M,0.514465,0.535543,0.441867


### Correlation by Sex

After separating the data by sex, the relationship between bodyweight and each lift becomes weaker than in the combined dataset.

For female lifters, squat has the strongest correlation with bodyweight ($r = 0.399$), followed by bench press ($r = 0.375$) and deadlift ($r = 0.362$). For male lifters, bench press has the strongest correlation ($r = 0.536$), followed by squat ($r = 0.514$) and deadlift ($r = 0.442$).

These results show that the lift most strongly associated with bodyweight differs by sex. The correlations within each sex are also lower than those observed when all lifters are combined. This suggests that part of the stronger relationship in the pooled dataset may reflect differences between male and female lifters rather than bodyweight alone.

Overall, these findings support including sex as an interactive filter in the final visualization rather than relying only on the combined population.


In [18]:
# Fit a separate least-squares regression line for each lift using the full dataset.
# The slope represents the estimated change in kilograms lifted per additional
# kilogram of bodyweight.
lifts = ["Squat", "Bench", "Deadlift"]

for lift in lifts:
    slope, intercept = np.polyfit(
        df["BodyweightKg"],
        df[lift],
        1
    )
    
    print(f"{lift}:")
    print(f"  Slope = {slope:.3f}")
    print(f"  Intercept = {intercept:.3f}")
    print()


Squat:
  Slope = 1.631
  Intercept = 26.830

Bench:
  Slope = 1.166
  Intercept = 5.836

Deadlift:
  Slope = 1.528
  Intercept = 61.168



### Regression Slopes for Bodyweight and Lift Performance

The regression slopes provide a different perspective from the correlation coefficients. Squat has the largest slope at approximately $1.631$, followed by deadlift at $1.528$ and bench press at $1.166$.

According to these fitted linear relationships, each additional kilogram of bodyweight is associated with approximately 1.63 kg more squat, 1.53 kg more deadlift, and 1.17 kg more bench press.

This differs from the correlation results. Bench press had the strongest overall correlation with bodyweight ($r = 0.597$), but squat has the steepest regression slope. This distinction is important because correlation measures how consistently two variables vary together, whereas slope measures the estimated change in lift performance associated with a one-kilogram change in bodyweight.

Therefore, if "scales best" refers to the amount of additional weight lifted as bodyweight increases, squat currently appears to scale the most. However, because the previous correlation analysis showed meaningful differences after separating lifters by sex, the regression slopes should also be examined separately by sex before drawing a final conclusion.

In [19]:
# Fit the same regressions separately for male and female lifters.
results = []

for sex in df["Sex"].unique():
    sex_data = df[df["Sex"] == sex]

    for lift in ["Squat", "Bench", "Deadlift"]:
        slope, intercept = np.polyfit(
            sex_data["BodyweightKg"],
            sex_data[lift],
            1
        )

        results.append({
            "Sex": sex,
            "Lift": lift,
            "Slope": slope,
            "Intercept": intercept
        })

sex_slopes = pd.DataFrame(results)

sex_slopes


,Sex,Lift,Slope,Intercept
0,M,Squat,1.258049,77.095066
1,M,Bench,0.854817,47.974918
2,M,Deadlift,1.030309,125.266957
3,F,Squat,0.682726,61.413461
4,F,Bench,0.350410,35.994881
5,F,Deadlift,0.588835,89.841663


In [20]:
# Pivot the slope results into a compact comparison table.
sex_slopes.pivot(
    index="Sex",
    columns="Lift",
    values="Slope"
).round(3)


Lift,Bench,Deadlift,Squat
Sex,,,
F,0.350,0.589,0.683
M,0.855,1.030,1.258


### Regression Slopes by Sex

When the regression analysis is separated by sex, squat has the steepest relationship with bodyweight for both male and female lifters.

For male lifters, each additional kilogram of bodyweight is associated with approximately 1.26 kg more squat, 1.03 kg more deadlift, and 0.85 kg more bench press. For female lifters, each additional kilogram of bodyweight is associated with approximately 0.68 kg more squat, 0.59 kg more deadlift, and 0.35 kg more bench press.

The ordering is therefore consistent for both groups: squat has the largest regression slope, followed by deadlift and then bench press. This provides evidence that squat scales most strongly with increasing bodyweight when "scaling" is defined as the estimated increase in absolute weight lifted per additional kilogram of bodyweight.

This result also illustrates the distinction between slope and correlation. Bench press previously showed the strongest correlation with bodyweight among male lifters, meaning that bench performance is more consistently associated with bodyweight. However, squat has the steeper regression slope, meaning that the predicted increase in absolute weight lifted is greater for squat as bodyweight increases.

Because the relationships differ between male and female lifters, sex will remain an interactive filter in the final visualization.

In [31]:
# --------------------------------------------------
# 1. Stratified sample for scatterplot
#    3,000 male + 3,000 female lifters
# --------------------------------------------------

male_sample = df[df["Sex"] == "M"].sample(
    n=3000,
    random_state=42
)

female_sample = df[df["Sex"] == "F"].sample(
    n=3000,
    random_state=42
)

df_sample = pd.concat(
    [male_sample, female_sample],
    ignore_index=True
)

# Convert sampled data to long format
df_long = df_sample.melt(
    id_vars=["Name", "Sex", "BodyweightKg"],
    value_vars=["Squat", "Bench", "Deadlift"],
    var_name="Lift",
    value_name="WeightKg"
)

alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

In [32]:
# --------------------------------------------------
# 2. Calculate regression statistics using FULL data
# --------------------------------------------------

regression_results = []
regression_lines = []

for sex_filter in ["All", "M", "F"]:

    # Select full dataset for each sex option
    if sex_filter == "All":
        group = df
    else:
        group = df[df["Sex"] == sex_filter]

    # Bodyweight range actually observed in this group
    x_min_group = group["BodyweightKg"].min()
    x_max_group = group["BodyweightKg"].max()

    for lift in ["Squat", "Bench", "Deadlift"]:

        slope, intercept = np.polyfit(
            group["BodyweightKg"],
            group[lift],
            1
        )

        # Save slope for numerical label
        regression_results.append({
            "SexFilter": sex_filter,
            "Lift": lift,
            "Slope": slope,
            "Intercept": intercept
        })

        # Create two endpoints for the FULL-data regression line
        regression_lines.append({
            "SexFilter": sex_filter,
            "Lift": lift,
            "BodyweightKg": x_min_group,
            "PredictedWeight": slope * x_min_group + intercept
        })

        regression_lines.append({
            "SexFilter": sex_filter,
            "Lift": lift,
            "BodyweightKg": x_max_group,
            "PredictedWeight": slope * x_max_group + intercept
        })


slope_df = pd.DataFrame(regression_results)
line_df = pd.DataFrame(regression_lines)


# --------------------------------------------------
# 3. Fixed axis ranges for fair comparisons
# --------------------------------------------------

x_min = df["BodyweightKg"].min()
x_max = df["BodyweightKg"].max()

y_max = df[["Squat", "Bench", "Deadlift"]].max().max()


# --------------------------------------------------
# 4. Interactive dropdowns
# --------------------------------------------------

lift_select = alt.param(
    name="Lift_Select",
    value="Squat",
    bind=alt.binding_select(
        options=["Squat", "Bench", "Deadlift"],
        name="Lift: "
    )
)

sex_select = alt.param(
    name="Sex_Select",
    value="All",
    bind=alt.binding_select(
        options=["All", "M", "F"],
        labels=["All", "Male", "Female"],
        name="Sex: "
    )
)


# --------------------------------------------------
# 5. Scatterplot using SAMPLE data
# --------------------------------------------------

points = (
    alt.Chart(df_long)
    .add_params(
        lift_select,
        sex_select
    )
    .transform_filter(
        alt.datum.Lift == lift_select
    )
    .transform_filter(
        (sex_select == "All") |
        (alt.datum.Sex == sex_select)
    )
    .mark_circle(
        size=25,
        opacity=0.20
    )
    .encode(
        x=alt.X(
            "BodyweightKg:Q",
            title="Bodyweight (kg)",
            scale=alt.Scale(
                domain=[x_min, x_max]
            )
        ),
        y=alt.Y(
            "WeightKg:Q",
            title="Lift Weight (kg)",
            scale=alt.Scale(
                domain=[0, y_max]
            )
        ),
        tooltip=[
            alt.Tooltip(
                "Name:N",
                title="Lifter"
            ),
            alt.Tooltip(
                "Sex:N",
                title="Sex"
            ),
            alt.Tooltip(
                "Lift:N",
                title="Lift"
            ),
            alt.Tooltip(
                "BodyweightKg:Q",
                title="Bodyweight (kg)",
                format=".1f"
            ),
            alt.Tooltip(
                "WeightKg:Q",
                title="Lift Weight (kg)",
                format=".1f"
            )
        ]
    )
)


# --------------------------------------------------
# 6. Regression line using FULL dataset statistics
# --------------------------------------------------

trend = (
    alt.Chart(line_df)
    .transform_filter(
        alt.datum.Lift == lift_select
    )
    .transform_filter(
        alt.datum.SexFilter == sex_select
    )
    .mark_line(
        size=3
    )
    .encode(
        x=alt.X(
            "BodyweightKg:Q",
            scale=alt.Scale(
                domain=[x_min, x_max]
            )
        ),
        y=alt.Y(
            "PredictedWeight:Q",
            scale=alt.Scale(
                domain=[0, y_max]
            )
        )
    )
)


# --------------------------------------------------
# 7. Dynamic numerical slope using FULL dataset
# --------------------------------------------------

slope_label = (
    alt.Chart(slope_df)
    .transform_filter(
        alt.datum.Lift == lift_select
    )
    .transform_filter(
        alt.datum.SexFilter == sex_select
    )
    .transform_calculate(
        Label="'Regression slope: ' + "
              "format(datum.Slope, '.3f') + ' kg/kg'"
    )
    .mark_text(
        align="left",
        baseline="top",
        fontSize=16,
        fontWeight="bold"
    )
    .encode(
        x=alt.value(15),
        y=alt.value(15),
        text="Label:N"
    )
)


# --------------------------------------------------
# 8. Final interactive visualization
# --------------------------------------------------

task1_chart = (
    points + trend + slope_label
).properties(
    width=750,
    height=500,
    title=alt.TitleParams(
        text="How Does Strength Scale with Bodyweight?",
        subtitle=[
            "Points show a stratified sample of 6,000 lifters.",
            "Regression line and slope are calculated using the full dataset."
        ]
    )
)

task1_chart

alt.LayerChart(...)

### Task 1 Findings

The visualization shows a positive relationship between bodyweight and absolute strength across squat, bench press, and deadlift. To compare how each lift scales with bodyweight, the regression slope represents the estimated increase in kilograms lifted associated with each additional kilogram of bodyweight.

When the data are separated by sex, **squat has the steepest regression slope for both male and female lifters**. For male lifters, the squat slope is approximately $1.258$, compared with $1.030$ for deadlift and $0.855$ for bench press. For female lifters, the squat slope is approximately $0.683$, compared with $0.589$ for deadlift and $0.350$ for bench press. Therefore, when "scaling" is defined as the increase in absolute weight lifted associated with increasing bodyweight, **squat scales the most with bodyweight in both groups**.

Another notable result is that the relationships are consistently steeper among male lifters than female lifters. For example, an additional kilogram of bodyweight is associated with approximately $1.258$ kg greater squat performance among male lifters, compared with approximately $0.683$ kg among female lifters. Because these data compare different lifters rather than tracking the same lifters as their bodyweight changes, these relationships should be interpreted as associations rather than causal effects of gaining bodyweight.

The combined-sex results are also particularly interesting. The regression slopes for the combined population are larger than the corresponding slopes calculated separately for males and females. This occurs because the pooled regression captures not only the relationship between bodyweight and strength within each sex, but also differences in average bodyweight and absolute strength between the two groups. As a result, combining the groups strengthens the apparent relationship between bodyweight and absolute strength.

This finding demonstrates why the interactive sex filter is important. Viewing only the combined population could lead to an incomplete interpretation of the relationship between bodyweight and strength. Allowing the user to switch between the combined, male, and female populations reveals how grouping the data can substantially affect the observed regression relationship.

For visualization performance, the scatterplot displays a reproducible stratified sample of 6,000 lifters (3,000 male and 3,000 female). However, the displayed regression lines and regression slopes are calculated using the complete cleaned dataset rather than the sample.

## Task 2: At what bodyweight does total strength begin to show diminishing returns?


In [35]:
# Task 2: Total strength by bodyweight and sex

# Create 5 kg bodyweight bins
df["BodyweightBin"] = np.floor(df["BodyweightKg"] / 5) * 5

# Calculate summary statistics for each Sex × Bodyweight bin
total_by_bw_sex = (
    df.groupby(["Sex", "BodyweightBin"])
      .agg(
          MedianTotal=("Total", "median"),
          MeanTotal=("Total", "mean"),
          Lifters=("Total", "count")
      )
      .reset_index()
)

# Keep only bins containing at least 100 lifters
total_by_bw_sex = total_by_bw_sex[
    total_by_bw_sex["Lifters"] >= 100
].reset_index(drop=True)

# Display results
total_by_bw_sex

,Sex,BodyweightBin,MedianTotal,MeanTotal,Lifters
0,F,40.0,190.00,192.492469,239
1,F,45.0,228.75,234.958411,900
2,F,50.0,260.00,260.933224,2342
3,F,55.0,280.00,282.284477,4445
4,F,60.0,287.50,290.831506,4217
5,F,65.0,303.75,305.730175,4514
6,F,70.0,315.00,317.696926,4252
7,F,75.0,310.00,312.405602,2085
8,F,80.0,327.50,330.665625,2510
9,F,85.0,330.00,333.013298,1507


In [36]:
# Task 2 exploratory visualization:
# Median Total vs. Bodyweight by Sex

sex_select_task2 = alt.param(
    name="Sex_Task2",
    value="M",
    bind=alt.binding_select(
        options=["M", "F"],
        labels=["Male", "Female"],
        name="Sex: "
    )
)

task2_exploration = (
    alt.Chart(total_by_bw_sex)
    .add_params(sex_select_task2)
    .transform_filter(
        alt.datum.Sex == sex_select_task2
    )
    .mark_line(
        point=True,
        strokeWidth=3
    )
    .encode(
        x=alt.X(
            "BodyweightBin:Q",
            title="Bodyweight (kg)",
            scale=alt.Scale(zero=False)
        ),
        y=alt.Y(
            "MedianTotal:Q",
            title="Median Total (kg)",
            scale=alt.Scale(zero=False)
        ),
        tooltip=[
            alt.Tooltip(
                "BodyweightBin:Q",
                title="Bodyweight (kg)"
            ),
            alt.Tooltip(
                "MedianTotal:Q",
                title="Median Total (kg)",
                format=".1f"
            ),
            alt.Tooltip(
                "MeanTotal:Q",
                title="Mean Total (kg)",
                format=".1f"
            ),
            alt.Tooltip(
                "Lifters:Q",
                title="Number of Lifters"
            )
        ]
    )
    .properties(
        width=750,
        height=500,
        title="How Does Total Strength Change with Bodyweight?"
    )
)

task2_exploration

alt.Chart(...)

In [37]:
# Calculate marginal change in median Total
# for each additional bodyweight bin

marginal_gains = total_by_bw_sex.copy()

# Ensure observations are ordered correctly
marginal_gains = marginal_gains.sort_values(
    ["Sex", "BodyweightBin"]
)

# Change in bodyweight from previous bin
marginal_gains["BodyweightChange"] = (
    marginal_gains.groupby("Sex")["BodyweightBin"].diff()
)

# Change in median Total from previous bin
marginal_gains["TotalChange"] = (
    marginal_gains.groupby("Sex")["MedianTotal"].diff()
)

# Additional Total per additional kg of bodyweight
marginal_gains["MarginalGain"] = (
    marginal_gains["TotalChange"] /
    marginal_gains["BodyweightChange"]
)

# Display rounded results
marginal_gains[
    [
        "Sex",
        "BodyweightBin",
        "MedianTotal",
        "TotalChange",
        "MarginalGain",
        "Lifters"
    ]
].round(2)

,Sex,BodyweightBin,MedianTotal,TotalChange,MarginalGain,Lifters
0,F,40.0,190.00,NaN,NaN,239
1,F,45.0,228.75,38.75,7.75,900
2,F,50.0,260.00,31.25,6.25,2342
3,F,55.0,280.00,20.00,4.00,4445
4,F,60.0,287.50,7.50,1.50,4217
5,F,65.0,303.75,16.25,3.25,4514
6,F,70.0,315.00,11.25,2.25,4252
7,F,75.0,310.00,-5.00,-1.00,2085
8,F,80.0,327.50,17.50,3.50,2510
9,F,85.0,330.00,2.50,0.50,1507


In [38]:
# --------------------------------------------------
# Task 2: Smooth marginal strength gains
# --------------------------------------------------

marginal_gains = marginal_gains.sort_values(
    ["Sex", "BodyweightBin"]
).copy()

# Calculate a centered 3-bin rolling average
# Each point summarizes approximately 15 kg of bodyweight
marginal_gains["SmoothedMarginalGain"] = (
    marginal_gains
    .groupby("Sex")["MarginalGain"]
    .transform(
        lambda x: x.rolling(
            window=3,
            center=True,
            min_periods=2
        ).mean()
    )
)

# Show the results separately for easier interpretation
female_marginal = (
    marginal_gains[
        marginal_gains["Sex"] == "F"
    ][
        [
            "BodyweightBin",
            "MedianTotal",
            "MarginalGain",
            "SmoothedMarginalGain",
            "Lifters"
        ]
    ]
    .round(2)
)

male_marginal = (
    marginal_gains[
        marginal_gains["Sex"] == "M"
    ][
        [
            "BodyweightBin",
            "MedianTotal",
            "MarginalGain",
            "SmoothedMarginalGain",
            "Lifters"
        ]
    ]
    .round(2)
)

print("FEMALE")
display(female_marginal)

print("\nMALE")
display(male_marginal)

FEMALE


,BodyweightBin,MedianTotal,MarginalGain,SmoothedMarginalGain,Lifters
0,40.0,190.00,NaN,NaN,239
1,45.0,228.75,7.75,7.00,900
2,50.0,260.00,6.25,6.00,2342
3,55.0,280.00,4.00,3.92,4445
4,60.0,287.50,1.50,2.92,4217
5,65.0,303.75,3.25,2.33,4514
6,70.0,315.00,2.25,1.50,4252
7,75.0,310.00,-1.00,1.58,2085
8,80.0,327.50,3.50,1.00,2510
9,85.0,330.00,0.50,1.00,1507



MALE


,BodyweightBin,MedianTotal,MarginalGain,SmoothedMarginalGain,Lifters
19,45.0,202.5,NaN,NaN,129
20,50.0,287.5,17.0,17.25,439
21,55.0,375.0,17.5,13.17,1501
22,60.0,400.0,5.0,10.67,1518
23,65.0,447.5,9.5,7.50,3642
24,70.0,487.5,8.0,6.00,7928
25,75.0,490.0,0.5,5.50,3841
26,80.0,530.0,8.0,4.00,9263
27,85.0,547.5,3.5,4.33,8189
28,90.0,555.0,1.5,3.33,5315


In [39]:
# ============================================================
# TASK 2 FINAL VISUALIZATION
# At what bodyweight does total strength show diminishing returns?
# ============================================================

import altair as alt
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Prepare Task 2 data
# ------------------------------------------------------------

task2_data = marginal_gains.copy()

# Keep only Male and Female
task2_data = task2_data[
    task2_data["Sex"].isin(["M", "F"])
].copy()


# ------------------------------------------------------------
# 2. Define approximate diminishing-return regions
#    based on our exploratory analysis
# ------------------------------------------------------------

diminishing_regions = pd.DataFrame({
    "Sex": ["F", "M"],
    "Start": [80, 105],
    "End": [90, 120]
})


# ------------------------------------------------------------
# 3. Interactive sex selector
# ------------------------------------------------------------

sex_select_task2 = alt.param(
    name="Sex_Task2",
    value="M",
    bind=alt.binding_select(
        options=["M", "F"],
        labels=["Male", "Female"],
        name="Sex: "
    )
)


# ------------------------------------------------------------
# 4. Shaded diminishing-return region
# ------------------------------------------------------------

region = (
    alt.Chart(diminishing_regions)
    .transform_filter(
        alt.datum.Sex == sex_select_task2
    )
    .mark_rect(
        opacity=0.15
    )
    .encode(
        x=alt.X(
            "Start:Q",
            scale=alt.Scale(domain=[40, 160])
        ),
        x2="End:Q"
    )
)


# ------------------------------------------------------------
# 5. Median Total line
# ------------------------------------------------------------

total_line = (
    alt.Chart(task2_data)
    .add_params(sex_select_task2)
    .transform_filter(
        alt.datum.Sex == sex_select_task2
    )
    .mark_line(
        point=True,
        strokeWidth=3
    )
    .encode(
        x=alt.X(
            "BodyweightBin:Q",
            title="Bodyweight (kg)",
            scale=alt.Scale(
                domain=[40, 160]
            )
        ),

        y=alt.Y(
            "MedianTotal:Q",
            title="Median Total (kg)",
            scale=alt.Scale(
                domain=[150, 700]
            )
        ),

        tooltip=[
            alt.Tooltip(
                "BodyweightBin:Q",
                title="Bodyweight",
                format=".0f"
            ),

            alt.Tooltip(
                "MedianTotal:Q",
                title="Median Total",
                format=".1f"
            ),

            alt.Tooltip(
                "SmoothedMarginalGain:Q",
                title="Smoothed Marginal Gain",
                format=".2f"
            ),

            alt.Tooltip(
                "Lifters:Q",
                title="Lifters",
                format=","
            )
        ]
    )
)


# ------------------------------------------------------------
# 6. Label the diminishing-return region
# ------------------------------------------------------------

region_label = (
    alt.Chart(diminishing_regions)
    .transform_filter(
        alt.datum.Sex == sex_select_task2
    )
    .transform_calculate(
        Midpoint="(datum.Start + datum.End) / 2",
        Label="'Approx. diminishing returns: ' + "
              "datum.Start + '–' + datum.End + ' kg'"
    )
    .mark_text(
        align="center",
        baseline="top",
        fontSize=14,
        fontWeight="bold",
        dy=10
    )
    .encode(
        x=alt.X(
            "Midpoint:Q",
            scale=alt.Scale(domain=[40, 160])
        ),

        y=alt.value(10),

        text="Label:N"
    )
)


# ------------------------------------------------------------
# 7. Final visualization
# ------------------------------------------------------------

task2_chart = (
    region +
    total_line +
    region_label
).properties(
    width=750,
    height=500,

    title=alt.TitleParams(
        text="Where Does Total Strength Begin to Show Diminishing Returns?",

        subtitle=[
            "Median total calculated in 5 kg bodyweight bins.",
            "Shaded region indicates the approximate onset of diminishing returns.",
            "Only bins containing at least 100 lifters are included."
        ]
    )
)

task2_chart

alt.LayerChart(...)

### Task 2 Findings — Where Does Total Strength Begin to Show Diminishing Returns?

The purpose of Task 2 was to investigate how total powerlifting performance changes as bodyweight increases and to determine whether there is a point at which additional bodyweight is associated with progressively smaller increases in strength. Initial exploration suggested that the relationship does not reach a single, clearly defined plateau. Instead, the data show a pattern of **diminishing returns**, where Total continues to increase at higher bodyweights but at a substantially slower and less consistent rate.

To make the overall trend easier to interpret, lifters were grouped into **5 kg bodyweight bins**, and the median Total was calculated for each bin. Median Total was used rather than the mean because it is less sensitive to unusually strong or weak performances. Bins containing fewer than 100 lifters were excluded to reduce the influence of bodyweight ranges with very small sample sizes.

The analysis was also separated by sex because the relationship between bodyweight and absolute strength differs substantially between male and female lifters. Among **female lifters**, median Total increases rapidly at lighter bodyweights but begins to flatten substantially around **80–90 kg**. Among **male lifters**, the rapid increase in Total continues to a higher bodyweight, with substantial diminishing returns becoming apparent around **105–120 kg**.

To further evaluate this pattern, the marginal strength gain was calculated as:

$$
\text{Marginal Strength Gain}
=
\frac{\Delta \text{Median Total}}
{\Delta \text{Bodyweight}}
$$

This represents the change in median Total associated with each additional kilogram of bodyweight between adjacent bodyweight bins. Because individual bins showed considerable variation, a **3-bin centered rolling average** was used to smooth the marginal gains across approximately 15 kg of bodyweight. The smoothed results supported the pattern visible in the median Total curves: marginal strength gains generally become smaller as bodyweight increases.

The shaded areas in the visualization therefore represent **approximate regions where substantial diminishing returns begin**, rather than exact biological or performance thresholds. For female lifters, this region is approximately **80–90 kg**, while for male lifters it is approximately **105–120 kg**. These ranges were identified from the flattening of the median Total curve together with the decline in smoothed marginal strength gains.

The interactive sex filter is important because it reveals that the onset of diminishing returns occurs at different bodyweights for male and female lifters. The tooltip also provides the median Total, smoothed marginal gain, and number of lifters within each bodyweight bin, allowing the viewer to examine both the strength trend and the amount of data supporting each point.

Overall, the results suggest that **absolute strength continues to increase with bodyweight, but the strength gained per additional kilogram of bodyweight decreases substantially at heavier bodyweights**. Therefore, the data support the presence of diminishing returns rather than a single hard plateau.

## Task 3: How do squat, bench press, and deadlift proportions differ between male and female lifters?


In [43]:
# ============================================================
# TASK 3
# How do lift proportions differ between male and female lifters?
# ============================================================

import pandas as pd

# Make a separate dataframe for Task 3
task3_df = df.copy()

# ------------------------------------------------------------
# 1. Calculate each lift as a percentage of Total
# ------------------------------------------------------------

task3_df["SquatShare"] = (
    task3_df["Squat"] / task3_df["Total"] * 100
)

task3_df["BenchShare"] = (
    task3_df["Bench"] / task3_df["Total"] * 100
)

task3_df["DeadliftShare"] = (
    task3_df["Deadlift"] / task3_df["Total"] * 100
)

# ------------------------------------------------------------
# 2. Verify that the percentages add to approximately 100%
# ------------------------------------------------------------

task3_df["ShareTotal"] = (
    task3_df["SquatShare"]
    + task3_df["BenchShare"]
    + task3_df["DeadliftShare"]
)

print("Share total check:")
print(task3_df["ShareTotal"].describe())


# ------------------------------------------------------------
# 3. Calculate typical lift proportions by sex
# ------------------------------------------------------------

lift_share_summary = (
    task3_df
    .groupby("Sex")
    .agg(
        MedianSquatShare=("SquatShare", "median"),
        MedianBenchShare=("BenchShare", "median"),
        MedianDeadliftShare=("DeadliftShare", "median"),

        MeanSquatShare=("SquatShare", "mean"),
        MeanBenchShare=("BenchShare", "mean"),
        MeanDeadliftShare=("DeadliftShare", "mean"),

        Lifters=("Total", "count")
    )
    .round(2)
)

print("\nLift proportions by sex:")
display(lift_share_summary)


# ------------------------------------------------------------
# 4. Difference between male and female median proportions
# ------------------------------------------------------------

male = lift_share_summary.loc["M"]
female = lift_share_summary.loc["F"]

comparison = pd.DataFrame({
    "Lift": ["Squat", "Bench", "Deadlift"],

    "Male Median %": [
        male["MedianSquatShare"],
        male["MedianBenchShare"],
        male["MedianDeadliftShare"]
    ],

    "Female Median %": [
        female["MedianSquatShare"],
        female["MedianBenchShare"],
        female["MedianDeadliftShare"]
    ]
})

comparison["Difference (M - F)"] = (
    comparison["Male Median %"]
    - comparison["Female Median %"]
)

comparison = comparison.round(2)

print("\nMale vs. Female comparison:")
display(comparison)

Share total check:
count    93490.000000
mean       100.000023
std          0.001995
min         99.908173
25%        100.000000
50%        100.000000
75%        100.000000
max        100.183824
Name: ShareTotal, dtype: float64

Lift proportions by sex:


,MedianSquatShare,MedianBenchShare,MedianDeadliftShare,MeanSquatShare,MeanBenchShare,MeanDeadliftShare,Lifters
Sex,,,,,,,
F,36.30,20.00,43.61,36.09,20.12,43.79,31300
M,35.61,23.26,41.06,35.47,23.38,41.15,62190



Male vs. Female comparison:


,Lift,Male Median %,Female Median %,Difference (M - F)
0,Squat,35.61,36.30,-0.69
1,Bench,23.26,20.00,3.26
2,Deadlift,41.06,43.61,-2.55


In [45]:
# ============================================================
# TASK 3 FINAL VISUALIZATION
# How Do Lift Proportions Differ Between Male and Female Lifters?
# ============================================================

import pandas as pd
import altair as alt

# ------------------------------------------------------------
# 1. Reshape lift-share data
# ------------------------------------------------------------

task3_long = task3_df.melt(
    id_vars=["Sex"],
    value_vars=["SquatShare", "BenchShare", "DeadliftShare"],
    var_name="Lift",
    value_name="Share"
)

task3_long["Lift"] = task3_long["Lift"].replace({
    "SquatShare": "Squat",
    "BenchShare": "Bench",
    "DeadliftShare": "Deadlift"
})

# Explicit lift order
lift_order = ["Squat", "Bench", "Deadlift"]

# ------------------------------------------------------------
# 2. Calculate mean contribution by sex
# ------------------------------------------------------------

task3_summary = (
    task3_long
    .groupby(["Sex", "Lift"], as_index=False)
    .agg(
        MeanShare=("Share", "mean"),
        Lifters=("Share", "count")
    )
)

task3_summary["SexLabel"] = task3_summary["Sex"].replace({
    "M": "Male",
    "F": "Female"
})

task3_summary["Lift"] = pd.Categorical(
    task3_summary["Lift"],
    categories=lift_order,
    ordered=True
)

task3_summary = task3_summary.sort_values(
    ["SexLabel", "Lift"]
)

task3_summary["MeanShare"] = task3_summary["MeanShare"].round(2)

task3_summary["Label"] = (
    task3_summary["MeanShare"]
    .map(lambda x: f"{x:.1f}%")
)

# ------------------------------------------------------------
# 3. Calculate start/end/midpoint for each stacked segment
#    This ensures percentage labels sit in the true center
# ------------------------------------------------------------

task3_summary["Start"] = (
    task3_summary
    .groupby("SexLabel", observed=True)["MeanShare"]
    .cumsum()
    - task3_summary["MeanShare"]
)

task3_summary["End"] = (
    task3_summary["Start"]
    + task3_summary["MeanShare"]
)

task3_summary["Midpoint"] = (
    task3_summary["Start"]
    + task3_summary["MeanShare"] / 2
)

# ------------------------------------------------------------
# 4. Stacked horizontal bars
# ------------------------------------------------------------

bars = (
    alt.Chart(task3_summary)
    .mark_bar(height=75)
    .encode(
        y=alt.Y(
            "SexLabel:N",
            title=None,
            sort=["Female", "Male"],
            axis=alt.Axis(labelFontSize=15)
        ),

        x=alt.X(
            "Start:Q",
            title="Percentage of Total (%)",
            scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(
                values=[0, 20, 40, 60, 80, 100],
                labelExpr="datum.value + '%'"
            )
        ),

        x2="End:Q",

        color=alt.Color(
            "Lift:N",
            title="Lift",
            sort=lift_order,
            scale=alt.Scale(
                domain=lift_order
            )
        ),

        tooltip=[
            alt.Tooltip(
                "SexLabel:N",
                title="Sex"
            ),
            alt.Tooltip(
                "Lift:N",
                title="Lift"
            ),
            alt.Tooltip(
                "MeanShare:Q",
                title="Share of Total (%)",
                format=".2f"
            ),
            alt.Tooltip(
                "Lifters:Q",
                title="Lifters",
                format=","
            )
        ]
    )
)

# ------------------------------------------------------------
# 5. Percentage labels centered within each segment
# ------------------------------------------------------------

labels = (
    alt.Chart(task3_summary)
    .mark_text(
        color="white",
        fontSize=14,
        fontWeight="bold"
    )
    .encode(
        y=alt.Y(
            "SexLabel:N",
            sort=["Female", "Male"]
        ),

        x=alt.X(
            "Midpoint:Q",
            scale=alt.Scale(domain=[0, 100])
        ),

        text="Label:N"
    )
)

# ------------------------------------------------------------
# 6. Final visualization
# ------------------------------------------------------------

task3_chart = (
    bars + labels
).properties(
    width=800,
    height=250,

    title=alt.TitleParams(
        text="How Do Lift Proportions Differ Between Male and Female Lifters?",
        subtitle=[
            "Average contribution of squat, bench press, and deadlift to total.",
            "Raw USAPL full-power performances; one highest-total performance per Name–Sex combination."
        ],
        fontSize=20,
        subtitleFontSize=13,
        anchor="middle"
    )
).configure_legend(
    orient="bottom",
    title=None,
    labelFontSize=13
).configure_view(
    stroke=None
)

task3_chart


alt.LayerChart(...)

### Task 3 Findings — How Do Lift Proportions Differ Between Male and Female Lifters?

The third task examines how the three competition lifts contribute to a lifter's overall total and whether this composition differs between male and female lifters. For each retained performance, squat, bench press, and deadlift were expressed as a percentage of the recorded total. The visualization displays the **mean contribution** of each lift so that the three components sum to approximately 100%.

The results show that **squat contributes a very similar proportion of total strength for male and female lifters**. Squat accounts for approximately **36.1% of the female total** and **35.5% of the male total**, a difference of less than one percentage point.

The larger differences occur in the relative contributions of the **bench press and deadlift**. Bench press represents approximately **23.4% of the male total**, compared with **20.1% for female lifters**. In contrast, deadlift accounts for approximately **43.8% of the female total**, compared with **41.1% for male lifters**.

Overall, the results suggest that the primary difference in lift composition between the sexes is not in the squat, but rather in the balance between **bench press and deadlift**. Male lifters in this dataset tend to derive a greater share of their total from the bench press, while female lifters tend to derive a greater share from the deadlift.

These results are descriptive and should not be interpreted as evidence that sex directly causes the observed differences. Other factors not represented in the visualization, such as training history, body proportions, weight class, and competition experience, may contribute to these patterns.


## Key Takeaways

- **Squat scales most strongly with bodyweight** within both male and female lifters when scaling is measured using the regression slope.
- **Absolute total strength shows diminishing returns at heavier bodyweights** rather than reaching a single hard plateau. The approximate onset occurs around **80–90 kg for women** and **105–120 kg for men** in this sample.
- **Lift composition differs by sex primarily through bench press and deadlift contribution.** Squat contributes a similar share of total performance for both groups, while men have a larger average bench share and women have a larger average deadlift share.

These findings are descriptive associations from the analyzed USAPL sample and should not be interpreted as causal physiological effects.


## Reproducibility Notes

The notebook expects a cleaned data file named `openpowerlifting_usapl_clean.csv` in the working directory. The original OpenPowerlifting download is not included in the repository because of its size.

For a public GitHub repository, include instructions in the README for obtaining the source data and reproducing the cleaning process. Interactive Altair charts can also be exported as standalone HTML files for browser-based viewing.
